# Machine learning

In [1]:
import pandas as pd
import numpy as np
from pycaret.classification import *

## Loading data

In [2]:
# Load encoded feature matrices saved during feature engineering
X_train = pd.read_csv('../data/features/X_train.csv')
X_test = pd.read_csv('../data/features/X_test.csv')

# Load target vectors — squeeze converts single-column DataFrame to Series
y_train = pd.read_csv('../data/features/y_train.csv').squeeze()
y_test = pd.read_csv('../data/features/y_test.csv').squeeze()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'y_test distribution:\n{y_test.value_counts(normalize=True).round(3)}')

X_train shape: (234436, 30)
X_test shape:  (58609, 30)
y_train distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64
y_test distribution:
citation_top_10_percent
0    0.82
1    0.18
Name: proportion, dtype: float64


## Combining train data for PyCaret

In [3]:
# PyCaret setup() requires a single DataFrame with features + target combined
# X_test and y_test are kept separate for final evaluation after modelling
train_data = X_train.copy()
train_data['target'] = y_train.values

print(f'Train data shape: {train_data.shape}')
print(f'Target distribution:\n{train_data["target"].value_counts()}')

Train data shape: (234436, 31)
Target distribution:
target
0    192194
1     42242
Name: count, dtype: int64


##  PyCaret setup

In [4]:
# Initialise PyCaret experiment
# - target: column to predict
# - test_data: holdout set — PyCaret will never touch this during training
# - fix_imbalance: applies SMOTE to training folds only to handle 82/18 class imbalance
# - session_id: random seed for reproducibility
# - index=False: resets index to RangeIndex to avoid duplicate index error
# - verbose: shows setup summary table
experiment = setup(
    data=train_data,
    target='target',
    test_data=pd.concat([X_test, y_test.rename('target')], axis=1).reset_index(drop=True),
    fix_imbalance=True,
    session_id=42,
    index=False,
    verbose=True
)

,Description,Value
0,Session id,42
1,Target,target
2,Target type,Binary
3,Original data shape,"(293045, 31)"
4,Transformed data shape,"(442997, 31)"
5,Transformed train set shape,"(384388, 31)"
6,Transformed test set shape,"(58609, 31)"
7,Numeric features,30
8,Preprocess,True
9,Imputation type,simple


## Compare models

In [5]:
# Compare all available classification models
# metric: optimise ranking by F1 score — appropriate for imbalanced classification
# sort: rank models by F1
# n_select: keep top 5 models for further analysis
best_models = compare_models(
    sort='F1',
    n_select=5,
    verbose=True
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.8190,0.8413,0.5961,0.4982,0.5428,0.4311,0.4338,21.0780
ada,Ada Boost Classifier,0.7842,0.8185,0.6460,0.4336,0.5189,0.3866,0.3994,6.0410
ridge,Ridge Classifier,0.7315,0.8123,0.7511,0.3771,0.5021,0.3449,0.3833,1.9700
lda,Linear Discriminant Analysis,0.7315,0.8123,0.7511,0.3771,0.5021,0.3449,0.3833,2.4010
lr,Logistic Regression,0.7336,0.8061,0.7362,0.3774,0.4990,0.3423,0.3778,13.0990
lightgbm,Light Gradient Boosting Machine,0.8457,0.8510,0.4246,0.6017,0.4978,0.4099,0.4186,6.7200
xgboost,Extreme Gradient Boosting,0.8470,0.8519,0.4076,0.6134,0.4897,0.4040,0.4157,3.8950
catboost,CatBoost Classifier,0.8481,0.8547,0.3853,0.6282,0.4776,0.3946,0.4108,37.7200
rf,Random Forest Classifier,0.8352,0.8309,0.4007,0.5596,0.4670,0.3727,0.3800,17.9420
knn,K Neighbors Classifier,0.7088,0.7602,0.7023,0.3476,0.4650,0.2951,0.3296,17.6930


## Model Comparison Results

15 classification models were compared using 10-fold stratified cross-validation 
with SMOTE applied to training folds to handle the 82/18 class imbalance.
Models are ranked by F1 score — the primary metric for imbalanced classification.

**Winner: Gradient Boosting Classifier (gbc)**
- F1 = 0.54 — best balance of precision and recall
- AUC = 0.84 — strong class separation ability
- Beats the dummy baseline (F1 = 0.00) significantly

**Key observations:**
- XGBoost and CatBoost achieve the highest AUC (0.85) but lower F1 (0.49, 0.48)
  — they are better at overall class separation but less balanced on precision/recall
- Accuracy is a misleading metric here — the dummy classifier scores 0.82 accuracy
  by always predicting the majority class, confirming F1 is the right primary metric
- No model exceeded F1 = 0.55 — expected given only metadata features are available
  at publication time, with no full text, author network or venue quality signals
- GBC is selected for hyperparameter tuning and SHAP analysis